In [ ]:
import numpy as np
import hyperspy.api as hs
import sys
sys.path.append('..')
import roi_tools
import pandas as pd # Highly recommended for viewing the final swept stats

# 1. Define your configurations to sweep through
# We will store them in a dictionary with a descriptive name as the key
configs = {
    "4 Atoms": {
        "dis": np.array([0, 0, -1, 1]),
        "djs": np.array([-1, 1, 0, 0]) * 2
    },
    "8 Atoms": {
        "dis": np.array([0, 0, -1, -1, -1, 1, 1, 1]),
        "djs": np.array([-1, 1, -1, 0, 1, -1, 0, 1]) * 2
    },
    "14 Atoms": {
        "dis": np.array([0, 0, -1, -1, -1, 1, 1, 1, -2, -2, -2, 2, 2, 2]),
        "djs": np.array([-1, 1, -1, 0, 1, -1, 0, 1, -1, 0, 1, -1, 0, 1]) * 2
    }
}

# 2. Prepare a dictionary to store the collected intensities for each config
collected_intensities = {name: [] for name in configs.keys()}
metric_str = 'mean_intensity'

# 3. Sweep through the files ONCE
print("Processing files...")
for iter_idx in range(30):
    path = f"../data/simulations/pristine/Iter{iter_idx}.npy"
    s = hs.load(path)
    
    # Do the heavy lifting (building the grid) only once per file
    roi = roi_tools.ROI(s)
    roi.build_grid_dict()
    roi.get_atom_types()
    
    # Sweep through all configurations on this currently loaded grid
    for config_name, config_arrays in configs.items():
        roi.set_vicinity_coords(config_arrays["dis"], config_arrays["djs"])
        roi.get_vicinity(metric=metric_str)
        roi.get_relative_vicinity(metric=metric_str)
        
        # --- NEW EXTRACTION LOGIC FOR Lu ONLY ---
        lu_intensities = []
        for (i, j), patch in np.ndenumerate(roi.grid):
            # Ensure the patch exists and is an Iron atom
            if patch is not None and getattr(patch, 'atom_type', None) == 'Lu':
                lu_intensities.append(roi.relative_vicinity[i, j])
        
        # Convert to numpy array and store it
        collected_intensities[config_name].append(np.array(lu_intensities))
        
    # Optional progress tracker
    if (iter_idx + 1) % 5 == 0:
        print(f"Finished {iter_idx + 1}/30 files")

# 4. Process the results into statistics
print("\nCalculating statistics...")
summary_stats = []

for config_name, intensities_list in collected_intensities.items():
    # Concatenate all 30 arrays for this config
    all_intensities = np.concatenate(intensities_list).astype(float)
    
    # Filter out NaNs (edges, invalid cells, etc.)
    all_intensities = all_intensities[~np.isnan(all_intensities)]
    
    # Calculate stats
    mean_val = np.mean(all_intensities)
    std_val = np.std(all_intensities)
    max_val = np.max(all_intensities)
    min_val = np.min(all_intensities)
    
    # Append to our summary list
    summary_stats.append({
        "Configuration": config_name,
        "Mean": mean_val,
        "Std Dev": std_val,
        "Max": max_val,
        "Min": min_val,
        "Valid Count": len(all_intensities)
    })

# 5. Display the results nicely using pandas
results_df = pd.DataFrame(summary_stats)
print("\n--- Sweep Results ---")
print(results_df.to_string(index=False))

In [ ]:
import numpy as np
import hyperspy.api as hs
import sys
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker 

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial'],
    'axes.linewidth': 1.5,
    'xtick.major.width': 1.5,
    'ytick.major.width': 1.5,
    'axes.labelsize': 18,     
    'xtick.labelsize': 14,    
    'ytick.labelsize': 14,
    'text.usetex': False 
})

# Initialize figure, colors, and bin settings
fig, axes = plt.subplots(1, 3, figsize=(12, 6), sharey=True)
color_orange = '#d62728'
color_blue = '#ff7f0e'
bin_width = 0.0015
custom_bins = np.arange(0.95, 1.05 + bin_width, bin_width)

for ax, (config_name, intensities_list) in zip(axes, collected_intensities.items()):
    all_intensities = np.concatenate(intensities_list).astype(float)
    all_intensities = all_intensities[~np.isnan(all_intensities)]
    
    mean_val = np.mean(all_intensities)
    std_val = np.std(all_intensities)
    max_val = np.max(all_intensities)
    
    # Plot data distributions and statistical indicators
    ax.hist(all_intensities, bins=custom_bins, color=color_blue, alpha=0.65, 
            label=r'$\mathrm{I_{cen}} \, / \langle \mathrm{I_{vic}} \rangle$ on pristine Lu', zorder=3, orientation='horizontal')
    
    ax.axhspan(ymin=mean_val - std_val, ymax=mean_val + std_val, color=color_orange, 
               alpha=0.2, label='±1 std.', zorder=2, linewidth=0)
    
    ax.axhline(y=mean_val, color=color_orange, linestyle='--', linewidth=1.2, 
               label='Mean', zorder=4)
    ax.axhline(y=max_val, color=color_orange, linestyle='-', linewidth=2, zorder=4, label='Max') 

    trans = ax.get_yaxis_transform()
    ax.text(0.98, max_val + 0.0005, f'{max_val:.4f}', color=color_orange, transform=trans, 
            va='bottom', ha='right', fontsize=14, fontweight='bold', zorder=5)
    ax.text(0.98, mean_val + 0.0005, f'{mean_val:.4f}', color=color_orange, transform=trans, 
            va='bottom', ha='right', fontsize=14, zorder=5)

    ax.set_title(config_name.replace("_", " "), fontsize=18, pad=10) 
    ax.set_xlim(0, 100)
    
    ax.tick_params(direction='in', top=False, right=False, length=6)
    ax.xaxis.set_major_locator(ticker.MaxNLocator(nbins=4, integer=True, prune='upper'))

# Apply axis labels with adjusted spacing and create unified legend
axes[1].set_xlabel('Counts')
axes[0].set_ylabel(r'$\mathrm{I_{cen}} \, / \langle \mathrm{I_{vic}} \rangle$', fontweight='bold')
axes[0].set_ylim(0.95, 1.05)

handles, labels = axes[-1].get_legend_handles_labels()
axes[-1].legend(handles, labels, loc='upper left', bbox_to_anchor=(1.02, 1.0), 
                frameon=False, fontsize=14)

fig.subplots_adjust(wspace=0.0, right=0.88, left=0.08, bottom=0.12, top=0.88) 
plt.show()

In [ ]:
import os
import numpy as np
import hyperspy.api as hs
import sys
sys.path.append('..')
import roi_tools

def sweep_single_atom_metrics(batch='FeLu_down', total=25, num_layers=25, num_iters=10, target=(6, 6)):
    """
    Sweeps through simulation files across layers and iterations to extract 
    the relative vicinity metric for a specific atom coordinate. Returns a 
    dictionary containing 2D arrays for each configuration.
    """
    configs = {
        "4 Atoms": {
            "dis": np.array([0, 0, -1, 1]),
            "djs": np.array([-1, 1, 0, 0]) * 2
        },
        "8 Atoms": {
            "dis": np.array([0, 0, -1, -1, -1, 1, 1, 1]),
            "djs": np.array([-1, 1, -1, 0, 1, -1, 0, 1]) * 2
        },
        "14 Atoms": {
            "dis": np.array([0, 0, -1, -1, -1, 1, 1, 1, -2, -2, -2, 2, 2, 2]),
            "djs": np.array([-1, 1, -1, 0, 1, -1, 0, 1, -1, 0, 1, -1, 0, 1]) * 2
        }
    }
    
    # Pre-allocate 2D arrays filled with NaNs to maintain shape if files are missing
    results = {
        name: np.full((num_layers, num_iters), np.nan) 
        for name in configs.keys()
    }
    
    for layer in range(num_layers):
        for iter_idx in range(num_iters):
            path = f"../data/simulations/{batch}/Layer{layer}_Total{total}_Iter{iter_idx}.npy"
            
            if not os.path.exists(path):
                print(f"File not found: {path}")
                continue
                
            s = hs.load(path)
            roi = roi_tools.ROI(s)
            roi.build_grid_dict()
            roi.get_atom_types()
            
            for config_name, config_arrays in configs.items():
                roi.set_vicinity_coords(config_arrays["dis"], config_arrays["djs"])
                roi.get_vicinity(metric='mean_intensity')
                roi.get_relative_vicinity(metric='mean_intensity')
                
                results[config_name][layer, iter_idx] = roi.relative_vicinity[target]
                
    return results

# Execute the extraction
atom_metrics_down = sweep_single_atom_metrics(batch='FeLu_down')

print("\nExtraction complete. Array shapes:")
for config, data_matrix in atom_metrics_down.items():
    print(f"{config}: {data_matrix.shape}")

In [ ]:
import os
import numpy as np
import hyperspy.api as hs
import sys
sys.path.append('..')
import roi_tools

def sweep_single_atom_metrics(batch='FeLu_up', total=25, num_layers=25, num_iters=20, target=(7, 6)):
    """
    Sweeps through simulation files across layers and iterations to extract 
    the relative vicinity metric for a specific atom coordinate. Returns a 
    dictionary containing 2D arrays for each configuration.
    """
    configs = {
        "4 Atoms": {
            "dis": np.array([0, 0, -1, 1]),
            "djs": np.array([-1, 1, 0, 0]) * 2
        },
        "8 Atoms": {
            "dis": np.array([0, 0, -1, -1, -1, 1, 1, 1]),
            "djs": np.array([-1, 1, -1, 0, 1, -1, 0, 1]) * 2
        },
        "14 Atoms": {
            "dis": np.array([0, 0, -1, -1, -1, 1, 1, 1, -2, -2, -2, 2, 2, 2]),
            "djs": np.array([-1, 1, -1, 0, 1, -1, 0, 1, -1, 0, 1, -1, 0, 1]) * 2
        }
    }
    
    # Pre-allocate 2D arrays filled with NaNs to maintain shape if files are missing
    results = {
        name: np.full((num_layers, num_iters), np.nan) 
        for name in configs.keys()
    }
    
    for layer in range(num_layers):
        for iter_idx in range(num_iters):
            path = f"../data/simulations/{batch}/Layer{layer}_Total{total}_Iter{iter_idx}.npy"
            
            if not os.path.exists(path):
                print(f"File not found: {path}")
                continue
                
            s = hs.load(path)
            roi = roi_tools.ROI(s)
            roi.build_grid_dict()
            roi.get_atom_types()
            
            for config_name, config_arrays in configs.items():
                roi.set_vicinity_coords(config_arrays["dis"], config_arrays["djs"])
                roi.get_vicinity(metric='mean_intensity')
                roi.get_relative_vicinity(metric='mean_intensity')
                
                results[config_name][layer, iter_idx] = roi.relative_vicinity[target]
                
    return results

# Execute the extraction
atom_metrics_up = sweep_single_atom_metrics(batch='FeLu_up')

print("\nExtraction complete. Array shapes:")
for config, data_matrix in atom_metrics_up.items():
    print(f"{config}: {data_matrix.shape}")

In [ ]:
import numpy as np

def concatenate_metrics(metrics_up, metrics_down, axis=1):
    """
    Concatenates the 'up' and 'down' metric dictionaries along the specified axis.
    Default axis=1 concatenates along the iterations dimension.
    """
    combined_metrics = {}
    
    for config in metrics_up.keys():
        combined_metrics[config] = np.concatenate(
            (metrics_up[config], metrics_down[config]), 
            axis=axis
        )
        
    return combined_metrics

atom_metrics = concatenate_metrics(atom_metrics_up, atom_metrics_down, axis=1)

print("\nConcatenated array shapes:")
for config, data_matrix in atom_metrics.items():
    print(f"{config}: {data_matrix.shape}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial'],
    'axes.linewidth': 1.5,
    'xtick.major.width': 1.5,
    'ytick.major.width': 1.5,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'xtick.major.size': 6,
    'ytick.major.size': 6,
    'axes.labelsize': 18,     
    'xtick.labelsize': 14,    
    'ytick.labelsize': 14
})

def plot_metric_comparison(pristine_dict, antisite_dict, metrics):
    """
    Plots a side-by-side error bar comparison of Pristine vs Defect data.
    """
    fig, ax = plt.subplots(figsize=(4.5, 5))
    
    color_pristine = '#ff7f0e'
    color_defect = "#D6204E"
    
    x_positions = np.arange(len(metrics))
    offset = 0.12 
    
    for i, metric in enumerate(metrics):
        norm_mean, norm_std = pristine_dict[metric]
        anti_mean, anti_std = antisite_dict[metric]
        
        ax.errorbar(x_positions[i] - offset, norm_mean, yerr=norm_std, 
                    fmt='D', color=color_pristine, markersize=7, capsize=4, 
                    elinewidth=1.5)
        
        ax.errorbar(x_positions[i] + offset, anti_mean, yerr=anti_std, 
                    fmt='^', color=color_defect, markersize=8, capsize=4, 
                    elinewidth=1.5)

    ax.set_xticks(x_positions)
    ax.set_xticklabels(metrics)
    ax.set_ylabel(r'$\mathrm{I_{cen}} \, / \langle \mathrm{I_{vic}} \rangle$', fontweight='bold')
    
    # Set limits and explicitly generate ticks to hide the first and last labels
    ax.set_ylim(0.96, 1.04)
    y_ticks = np.arange(0.96, 1.041, 0.02)
    ax.set_yticks(y_ticks)
    
    y_labels = [f"{tick:.2f}" for tick in y_ticks]
    y_labels[0] = ""  
    y_labels[-1] = "" 
    ax.set_yticklabels(y_labels)
    ax.tick_params(axis='x', bottom=False, top=False)
    
    # Anchor x-limits to enforce equal width across all 3 configuration regions
    ax.set_xlim(-0.5, len(metrics) - 0.5)
    
    for x_val in [0.5, 1.5]:
        ax.axvline(x=x_val, color='black', linestyle='-', linewidth=1.5, zorder=0)
        
    legend_handles = [
        Line2D([0], [0], marker='^', color='w', markerfacecolor=color_defect, markersize=8),
        Line2D([0], [0], marker='D', color='w', markerfacecolor=color_pristine, markersize=8)
    ]
    
    # Tucked the legend slightly closer to the plot's right edge
    ax.legend(legend_handles, ['Defect', 'Pristine'], loc='upper left', 
              bbox_to_anchor=(1.02, 1.0), frameon=False, fontsize=12)
    
    fig.subplots_adjust(right=0.78, left=0.15, bottom=0.15, top=0.9)
    plt.show()

def main():
    """
    Extracts stats from both the pristine and antisite runs, formats them, 
    and triggers the final comparison plot.
    """
    pristine_data = {stat["Configuration"]: (stat["Mean"], stat["Std Dev"]) for stat in summary_stats}
    
    antisite_data = {}
    for config_name, data_matrix in atom_metrics.items():
        valid_data = data_matrix[~np.isnan(data_matrix)]
        antisite_data[config_name] = (np.mean(valid_data), np.std(valid_data))
        
    metrics_list = ["4 Atoms", "8 Atoms", "14 Atoms"]
    plot_metric_comparison(pristine_data, antisite_data, metrics_list)

main()

In [ ]:
import numpy as np
import hyperspy.api as hs
import sys
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker 

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial'],
    'axes.linewidth': 1.5,
    'xtick.major.width': 1.5,
    'ytick.major.width': 1.5,
    'axes.labelsize': 18,     
    'xtick.labelsize': 14,    
    'ytick.labelsize': 14,
    'text.usetex': False 
})

fig, axes = plt.subplots(1, 3, figsize=(12, 6), sharey=True)

color_pristine_hist = "#ff7f0e"  
color_pristine_stat = "#da6c0c"  
color_defect_hist = '#D6204E'  
color_defect_stat = "#9D0229"  

bin_width = 0.0015
custom_bins = np.arange(0.95, 1.04 + bin_width, bin_width)

for ax, (config_name, intensities_list) in zip(axes, collected_intensities.items()):
    all_intensities = np.concatenate(intensities_list).astype(float)
    all_intensities = all_intensities[~np.isnan(all_intensities)]
    
    norm_mean = np.mean(all_intensities)
    norm_std = np.std(all_intensities)
    norm_min = np.min(all_intensities)
    
    ax.hist(all_intensities, bins=custom_bins, color=color_pristine_hist, alpha=0.6, 
            label=r'$\mathrm{I_{cen}} \, / \langle \mathrm{I_{vic}} \rangle$ on pristine Lu', zorder=3, orientation='horizontal')
    
#     ax.axhspan(ymin=norm_mean - norm_std, ymax=norm_mean + norm_std, color=color_pristine_stat, 
#                alpha=0.2, label='Pristine ±1 std.', zorder=2, linewidth=0)
    ax.axhline(y=norm_mean, color=color_pristine_stat, linestyle='--', linewidth=1.2, 
               label='Pristine Mean', zorder=4)
    ax.axhline(y=norm_min, color=color_pristine_stat, linestyle='-', linewidth=2, 
               label='Pristine Min', zorder=4) 

    # Apply text annotations pushed to x=0.98
    trans = ax.get_yaxis_transform()
    ax.text(0.98, norm_min - 0.0013, f'{norm_min:.4f}', color=color_pristine_stat, transform=trans, 
            va='top', ha='right', fontsize=14, fontweight='bold', zorder=5)
    ax.text(0.98, norm_mean + 0.0005, f'{norm_mean:.4f}', color=color_pristine_stat, transform=trans, 
            va='bottom', ha='right', fontsize=14, zorder=5)

    if config_name in atom_metrics:
        defect_matrix = atom_metrics[config_name]
        defect_intensities = defect_matrix[~np.isnan(defect_matrix)]
        
        if len(defect_intensities) > 0:
            def_mean = np.mean(defect_intensities)
            def_std = np.std(defect_intensities)
            
            ax.hist(defect_intensities, bins=custom_bins, color=color_defect_hist, alpha=0.6, 
                    label=r'$\mathrm{I_{cen}} \, / \langle \mathrm{I_{vic}} \rangle$ on defect Lu', zorder=4, orientation='horizontal')
            
        #     ax.axhspan(ymin=def_mean - def_std, ymax=def_mean + def_std, color=color_defect_stat, 
        #                alpha=0.2, label='Defect ±1 std.', zorder=2, linewidth=0)
            ax.axhline(y=def_mean, color=color_defect_stat, linestyle='--', linewidth=1.2, 
                       label='Defect Mean', zorder=5)
                       
            # Mirror the same text label styling for the defect distribution
            ax.text(0.98, def_mean - 0.0013, f'{def_mean:.4f}', color=color_defect_stat, transform=trans, 
                    va='top', ha='right', fontsize=14, zorder=5)

    ax.set_title(config_name.replace("_", " "), fontsize=18, pad=10) 
    ax.tick_params(direction='in', top=False, right=False, length=6)
    ax.xaxis.set_major_locator(ticker.MaxNLocator(nbins=4, integer=True, prune='upper'))


axes[1].set_xlabel('Counts')
axes[0].set_ylabel(r'$\mathrm{I_{cen}} \, / \langle \mathrm{I_{vic}} \rangle$', fontweight='bold')
axes[0].set_ylim(0.95, 1.04)

axes[0].set_xlim(0, 140)
axes[1].set_xlim(0, 120)
axes[2].set_xlim(0, 90)

# Clean up duplicate legend entries from the loop
handles, labels = axes[-1].get_legend_handles_labels()
by_label = dict(zip(labels, handles))

axes[-1].legend(by_label.values(), by_label.keys(), loc='upper left', bbox_to_anchor=(1.02, 1.0), 
                frameon=False, fontsize=12)

fig.subplots_adjust(wspace=0.0, right=0.75, left=0.08, bottom=0.12, top=0.88) 
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial'],
    'axes.linewidth': 1.5,
    'xtick.major.width': 1.5,
    'ytick.major.width': 1.5,
    'axes.labelsize': 18,     
    'xtick.labelsize': 14,    
    'ytick.labelsize': 14,
    'text.usetex': False 
})

def plot_layer_metrics(metrics_dict, pristine_dict=None):
    """
    Generates a 1x3 subplot figure displaying scatter points, means,
    and standard deviations for each layer across three configurations.
    Includes a cutoff line based on the maximum pristine intensity and a legend.
    """
    fig, axes = plt.subplots(1, 3, figsize=(15, 6), sharey=True)
    color_scatter = "#ffc18b"
    color_mean = '#ff7f0e'
    color_cutoff = "#da6c0c"

    for ax, (config_name, data_matrix) in zip(axes, metrics_dict.items()):
        num_layers = data_matrix.shape[0]
        
        for layer_idx in range(num_layers):
            layer_data = data_matrix[layer_idx, :]
            valid_data = layer_data[~np.isnan(layer_data)]
            
            if len(valid_data) == 0:
                continue
                
            ax.scatter(np.full_like(valid_data, layer_idx), valid_data, 
                       color=color_scatter, alpha=0.6, s=20, zorder=1, 
                       label='Defect atoms')
            
            mean_val = np.mean(valid_data)
            std_val = np.std(valid_data)
            
            ax.errorbar(layer_idx, mean_val, yerr=std_val, fmt='o', 
                        color=color_mean, capsize=3, zorder=2,
                        label='Mean ± 1 std.')
            
        if pristine_dict is not None and config_name in pristine_dict:
            all_intensities = np.concatenate(pristine_dict[config_name]).astype(float)
            all_intensities = all_intensities[~np.isnan(all_intensities)]
            norm_min = np.min(all_intensities)
            
            ax.axhline(y=norm_min, color=color_cutoff, linestyle='-', linewidth=2, 
                       zorder=4, label='Pristine Min')
            
            trans = ax.get_yaxis_transform()
            ax.text(0.98, norm_min - 0.0013, f'{norm_min:.4f}', color=color_cutoff, 
                    transform=trans, va='top', ha='right', fontsize=14, 
                    fontweight='bold', zorder=5)
            
        ax.set_title(config_name, fontsize=18, pad=10)
        ax.set_xlim(-1, num_layers)
        ax.tick_params(direction='in', top=False, right=False, length=6)
        ax.xaxis.set_major_locator(ticker.MaxNLocator(nbins=5, integer=True))

    axes[1].set_xlabel('Layer Index (k)')
    axes[0].set_ylabel(r'$\mathrm{I_{cen}} \, / \langle \mathrm{I_{vic}} \rangle$', fontweight='bold')
    axes[0].set_ylim(0.95, 1.04)
    
    # Collect deduplicated labels and force 'Pristine Min' to the end of the legend
    handles, labels = [], []
    for ax in axes:
        h, l = ax.get_legend_handles_labels()
        handles.extend(h)
        labels.extend(l)
        
    by_label = dict(zip(labels, handles))
    ordered_labels = [lbl for lbl in by_label.keys() if lbl != 'Pristine Min']
    if 'Pristine Min' in by_label:
        ordered_labels.append('Pristine Min')
        
    ordered_handles = [by_label[lbl] for lbl in ordered_labels]
    
    axes[-1].legend(ordered_handles, ordered_labels, loc='upper left', 
                    bbox_to_anchor=(1.02, 1.0), frameon=False, fontsize=12)
    
    fig.subplots_adjust(wspace=0.0, right=0.82, left=0.08, bottom=0.12, top=0.88) 
    plt.show()

plot_layer_metrics(atom_metrics, collected_intensities)

In [ ]:
import numpy as np

def calculate_total_below_cutoff_ratio(metrics_dict, pristine_dict):
    """
    Calculates the overall ratio of atoms below the pristine minimum cutoff
    across all layers combined for each configuration.
    """
    results = {}

    for config_name, data_matrix in metrics_dict.items():
        if pristine_dict is None or config_name not in pristine_dict:
            continue
            
        all_intensities = np.concatenate(pristine_dict[config_name]).astype(float)
        all_intensities = all_intensities[~np.isnan(all_intensities)]
        norm_min = np.min(all_intensities)

        all_layer_data = data_matrix.flatten()
        valid_data = all_layer_data[~np.isnan(all_layer_data)]
        
        total_atoms = len(valid_data)
        if total_atoms == 0:
            results[config_name] = {'cutoff': norm_min, 'ratio': None, 'below': 0, 'total': 0}
            continue
            
        total_atoms_below = np.sum(valid_data < norm_min)
        overall_ratio = total_atoms_below / total_atoms
            
        results[config_name] = {
            'cutoff': norm_min,
            'ratio': overall_ratio,
            'below': total_atoms_below,
            'total': total_atoms
        }

    return results

overall_ratios = calculate_total_below_cutoff_ratio(atom_metrics, collected_intensities)

for config, data in overall_ratios.items():
    if data['ratio'] is not None:
        print(f"{config} (Cutoff: {data['cutoff']:.4f}):")
        print(f"  {data['below']} out of {data['total']} atoms below cutoff ({data['ratio']:.2%})\n")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial'],
    'axes.linewidth': 1.5,
    'xtick.major.width': 1.5,
    'ytick.major.width': 1.5,
    'axes.labelsize': 18,     
    'xtick.labelsize': 14,    
    'ytick.labelsize': 14,
    'text.usetex': False 
})

def plot_layer_metrics(metrics_dict, pristine_dict=None):
    """
    Generates a 1x3 subplot figure displaying scatter points, means,
    and standard deviations for each layer across three configurations.
    Includes a cutoff line based on the maximum pristine intensity and a legend.
    """
    fig, axes = plt.subplots(1, 3, figsize=(15, 6), sharey=True)
    color_scatter = "#ffc18b"
    color_mean = '#ff7f0e'
    color_cutoff = "#da6c0c"

    for ax, (config_name, data_matrix) in zip(axes, metrics_dict.items()):
        num_layers = data_matrix.shape[0]
        
        for layer_idx in range(num_layers):
            layer_data = data_matrix[layer_idx, :]
            valid_data = layer_data[~np.isnan(layer_data)]
            
            if len(valid_data) == 0:
                continue
                
            ax.scatter(np.full_like(valid_data, layer_idx), valid_data, 
                       color=color_scatter, alpha=0.6, s=20, zorder=1, 
                       label='Defect atoms')
            
            mean_val = np.mean(valid_data)
            std_val = np.std(valid_data)
            
            ax.errorbar(layer_idx, mean_val, yerr=std_val, fmt='o', 
                        color=color_mean, capsize=3, zorder=2,
                        label='Mean ± 1 std.')
            
        if pristine_dict is not None and config_name in pristine_dict:
            all_intensities = np.concatenate(pristine_dict[config_name]).astype(float)
            all_intensities = all_intensities[~np.isnan(all_intensities)]
            norm_min = np.min(all_intensities)
            
            ax.axhline(y=norm_min, color=color_cutoff, linestyle='-', linewidth=2, 
                       zorder=4, label='Pristine Min')
            
            trans = ax.get_yaxis_transform()
            ax.text(0.98, norm_min - 0.0013, f'{norm_min:.4f}', color=color_cutoff, 
                    transform=trans, va='top', ha='right', fontsize=14, 
                    fontweight='bold', zorder=5)
            
        ax.set_title(config_name, fontsize=18, pad=10)
        ax.set_xlim(-1, num_layers)
        ax.tick_params(direction='in', top=False, right=False, length=6)
        ax.xaxis.set_major_locator(ticker.MaxNLocator(nbins=5, integer=True))

    axes[1].set_xlabel('Layer Index (k)')
    axes[0].set_ylabel(r'$\mathrm{I_{cen}} \, / \langle \mathrm{I_{vic}} \rangle$', fontweight='bold')
    axes[0].set_ylim(0.95, 1.04)
    
    # Collect deduplicated labels and force 'Pristine Min' to the end of the legend
    handles, labels = [], []
    for ax in axes:
        h, l = ax.get_legend_handles_labels()
        handles.extend(h)
        labels.extend(l)
        
    by_label = dict(zip(labels, handles))
    ordered_labels = [lbl for lbl in by_label.keys() if lbl != 'Pristine Min']
    if 'Pristine Min' in by_label:
        ordered_labels.append('Pristine Min')
        
    ordered_handles = [by_label[lbl] for lbl in ordered_labels]
    
    axes[-1].legend(ordered_handles, ordered_labels, loc='upper left', 
                    bbox_to_anchor=(1.02, 1.0), frameon=False, fontsize=12)
    
    fig.subplots_adjust(wspace=0.0, right=0.82, left=0.08, bottom=0.12, top=0.88) 
    plt.show()

plot_layer_metrics(atom_metrics_up, collected_intensities)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial'],
    'axes.linewidth': 1.5,
    'xtick.major.width': 1.5,
    'ytick.major.width': 1.5,
    'axes.labelsize': 18,     
    'xtick.labelsize': 14,    
    'ytick.labelsize': 14,
    'text.usetex': False 
})

def plot_layer_metrics(metrics_dict, pristine_dict=None):
    """
    Generates a 1x3 subplot figure displaying scatter points, means,
    and standard deviations for each layer across three configurations.
    Includes a cutoff line based on the maximum pristine intensity and a legend.
    """
    fig, axes = plt.subplots(1, 3, figsize=(15, 6), sharey=True)
    color_scatter = "#ffc18b"
    color_mean = '#ff7f0e'
    color_cutoff = "#da6c0c"

    for ax, (config_name, data_matrix) in zip(axes, metrics_dict.items()):
        num_layers = data_matrix.shape[0]
        
        for layer_idx in range(num_layers):
            layer_data = data_matrix[layer_idx, :]
            valid_data = layer_data[~np.isnan(layer_data)]
            
            if len(valid_data) == 0:
                continue
                
            ax.scatter(np.full_like(valid_data, layer_idx), valid_data, 
                       color=color_scatter, alpha=0.6, s=20, zorder=1, 
                       label='Defect atoms')
            
            mean_val = np.mean(valid_data)
            std_val = np.std(valid_data)
            
            ax.errorbar(layer_idx, mean_val, yerr=std_val, fmt='o', 
                        color=color_mean, capsize=3, zorder=2,
                        label='Mean ± 1 std.')
            
        if pristine_dict is not None and config_name in pristine_dict:
            all_intensities = np.concatenate(pristine_dict[config_name]).astype(float)
            all_intensities = all_intensities[~np.isnan(all_intensities)]
            norm_min = np.min(all_intensities)
            
            ax.axhline(y=norm_min, color=color_cutoff, linestyle='-', linewidth=2, 
                       zorder=4, label='Pristine Min')
            
            trans = ax.get_yaxis_transform()
            ax.text(0.98, norm_min - 0.0013, f'{norm_min:.4f}', color=color_cutoff, 
                    transform=trans, va='top', ha='right', fontsize=14, 
                    fontweight='bold', zorder=5)
            
        ax.set_title(config_name, fontsize=18, pad=10)
        ax.set_xlim(-1, num_layers)
        ax.tick_params(direction='in', top=False, right=False, length=6)
        ax.xaxis.set_major_locator(ticker.MaxNLocator(nbins=5, integer=True))

    axes[1].set_xlabel('Layer Index (k)')
    axes[0].set_ylabel(r'$\mathrm{I_{cen}} \, / \langle \mathrm{I_{vic}} \rangle$', fontweight='bold')
    axes[0].set_ylim(0.95, 1.04)
    
    # Collect deduplicated labels and force 'Pristine Min' to the end of the legend
    handles, labels = [], []
    for ax in axes:
        h, l = ax.get_legend_handles_labels()
        handles.extend(h)
        labels.extend(l)
        
    by_label = dict(zip(labels, handles))
    ordered_labels = [lbl for lbl in by_label.keys() if lbl != 'Pristine Min']
    if 'Pristine Min' in by_label:
        ordered_labels.append('Pristine Min')
        
    ordered_handles = [by_label[lbl] for lbl in ordered_labels]
    
    axes[-1].legend(ordered_handles, ordered_labels, loc='upper left', 
                    bbox_to_anchor=(1.02, 1.0), frameon=False, fontsize=12)
    
    fig.subplots_adjust(wspace=0.0, right=0.82, left=0.08, bottom=0.12, top=0.88) 
    plt.show()

plot_layer_metrics(atom_metrics_down, collected_intensities)